# LAB | Relevance Scoring and Rerankers

This notebook builds a basic RAG retrieval pipeline, then compares retrieval quality before and after metadata filtering and reranking.

## Step 1: Data Loading

Load the Trustworthy AI transcript and the PDF document, then prepare them for chunking and retrieval.

In [1]:
from pathlib import Path
from collections import Counter
import os

import numpy as np
from dotenv import load_dotenv
from PyPDF2 import PdfReader
from openai import OpenAI

DATA_DIR = Path("data")

print(list(DATA_DIR.iterdir()))

[WindowsPath('data/hleg_ethics_guidelines_for_trustworthy_ai-en.pdf'), WindowsPath('data/trustworthy_ai_transcript.txt')]


In [2]:
transcript_path = DATA_DIR / "trustworthy_ai_transcript.txt"

with open(transcript_path, "r", encoding="utf-8") as f:
    transcript_text = f.read()

print(f"Transcript length: {len(transcript_text):,} characters")
print(transcript_text[:500])

Transcript length: 18,622 characters
0:00: So imagine for a second you're driving across a, I don't know, a massive suspension bridge. 
 0:06: You don't pull over halfway across, get out, and demand to see the blueprints, right? 
 0:10: You don't, , interview the welding crew. 
 0:12: No, you just, you trust it. 
 0:14: You just drive. 
 0:15: You trust the bridge. 
 0:15: You trust the engineering standards, the inspection. 
 0:18: The laws that say this thing won't fail, right? 
 0:20: It's trust in the infrastructure. 
 0:22: It


In [3]:
pdf_path = DATA_DIR / "hleg_ethics_guidelines_for_trustworthy_ai-en.pdf"

reader = PdfReader(pdf_path)

pdf_text = ""

for page in reader.pages:
    pdf_text += page.extract_text() + "\n"

print(f"PDF length: {len(pdf_text):,} characters")
print(pdf_text[:500])

PDF length: 158,079 characters
 
  
 
 
 
INDEPENDENT  
HIGH-LEVEL EXPERT GROUP ON  
ARTIFICIAL INTELLIGENCE  
SET UP BY THE EUROPEAN COMMISSION  
 
 
 
 
 
 
ETHICS GUIDELINES  
FOR TRUSTWORTHY AI 
 
 
 
  
 
ETHICS GUIDELINES  FOR  TRUSTWORTHY  AI 
 
High -Level Expert Group on Artificial Intelligence  
 
 
 
 
 
 
 
 
This document was written by the High -Level Expert Group on AI (AI HLEG) . The members of the AI HLEG 
named in this document support the overall framework for Trustworthy  AI put forward in these Guidelines


In [4]:
documents = [
    {
        "text": transcript_text,
        "source": "transcript"
    },
    {
        "text": pdf_text,
        "source": "pdf"
    }
]

for doc in documents:
    print(doc["source"], len(doc["text"]))

transcript 18622
pdf 158079


In [5]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = []

for doc in documents:
    text = doc["text"]

    for i in range(0, len(text), CHUNK_SIZE - CHUNK_OVERLAP):
        chunk_text = text[i:i + CHUNK_SIZE]

        chunks.append({
            "text": chunk_text,
            "source": doc["source"]
        })

print(f"Total chunks: {len(chunks)}")

for chunk in chunks[:3]:
    print(chunk["source"], len(chunk["text"]))

Total chunks: 222
transcript 1000
transcript 1000
transcript 1000


In [6]:
from collections import Counter

source_counts = Counter(chunk["source"] for chunk in chunks)

print(source_counts)

Counter({'pdf': 198, 'transcript': 24})


In [7]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print("Key loaded:", api_key is not None)

Key loaded: True


In [8]:
client = OpenAI(api_key=api_key)

print("OpenAI client ready")

OpenAI client ready


In [9]:
test_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks[0]["text"]
)

print(len(test_embedding.data[0].embedding))

1536


In [10]:
embeddings = []

for i, chunk in enumerate(chunks):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk["text"]
    )

    embeddings.append(response.data[0].embedding)

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/{len(chunks)}")

print(f"Finished: {len(embeddings)} embeddings")

Processed 25/222
Processed 50/222
Processed 75/222
Processed 100/222
Processed 125/222
Processed 150/222
Processed 175/222
Processed 200/222
Finished: 222 embeddings


In [11]:
embeddings_array = np.array(embeddings)

print(embeddings_array.shape)

(222, 1536)


In [12]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [13]:
def retrieve(query, top_k=5):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding

    scores = []

    for i, embedding in enumerate(embeddings):
        score = cosine_similarity(query_embedding, embedding)

        scores.append({
            "chunk_index": i,
            "score": score,
            "source": chunks[i]["source"],
            "text": chunks[i]["text"]
        })

    scores.sort(key=lambda x: x["score"], reverse=True)

    return scores[:top_k]

In [14]:
results = retrieve("What are the requirements for trustworthy AI?", top_k=3)

for result in results:
    print(f"Source: {result['source']}")
    print(f"Score: {result['score']:.4f}")
    print(result["text"][:300])
    print("-" * 50)

Source: pdf
Score: 0.7614
 of Trustworthy  AI, via a list of seven 
requirements that should be met, building on the principles outlined in Chapter I. In addition, available  technical 
and non -technical methods are introduced for the implementation of these requirements throughout the AI  
system’s life cycle .  
 
1. Requ
--------------------------------------------------
Source: pdf
Score: 0.7125
the trustworthiness of the AI system itself, but requires a holistic and systemic approach , encompassing  the 
trustworthiness of all actors and processes that are part of the system’s socio -technical context throughout its 
entire life cycle.  
Trustworthy  AI has three components , which should 
--------------------------------------------------
Source: pdf
Score: 0.7057
ith an AI system.  
 Facilitate the traceability and auditability of AI systems, particularly in critical contexts or situations.  
 Involve  stakeholders throughout the AI system ’s life cycle . Foster  training  an

In [15]:
def retrieve_by_source(query, source_filter, top_k=5):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding

    scores = []

    for i, embedding in enumerate(embeddings):

        if chunks[i]["source"] != source_filter:
            continue

        score = cosine_similarity(query_embedding, embedding)

        scores.append({
            "chunk_index": i,
            "score": score,
            "source": chunks[i]["source"],
            "text": chunks[i]["text"]
        })

    scores.sort(key=lambda x: x["score"], reverse=True)

    return scores[:top_k]

In [16]:
results = retrieve_by_source(
    "What are the requirements for trustworthy AI?",
    source_filter="pdf",
    top_k=3
)

for result in results:
    print(result["source"])
    print(f"Score: {result['score']:.4f}")
    print(result["text"][:200])
    print("-" * 50)

pdf
Score: 0.7614
 of Trustworthy  AI, via a list of seven 
requirements that should be met, building on the principles outlined in Chapter I. In addition, available  technical 
and non -technical methods are introduce
--------------------------------------------------
pdf
Score: 0.7125
the trustworthiness of the AI system itself, but requires a holistic and systemic approach , encompassing  the 
trustworthiness of all actors and processes that are part of the system’s socio -technic
--------------------------------------------------
pdf
Score: 0.7057
ith an AI system.  
 Facilitate the traceability and auditability of AI systems, particularly in critical contexts or situations.  
 Involve  stakeholders throughout the AI system ’s life cycle . Fo
--------------------------------------------------


In [17]:
results = retrieve_by_source(
    "What are the requirements for trustworthy AI?",
    source_filter="transcript",
    top_k=3
)

for result in results:
    print(result["source"])
    print(f"Score: {result['score']:.4f}")
    print(result["text"][:200])
    print("-" * 50)

transcript
Score: 0.6168
enerative AI explosion we have today. 
 1:22: So why are we blowing the dust off this specific report? 
 1:25: Because it's not dusty. 
 1:26: It's the foundation. 
 1:28: Everything we see now, like 
--------------------------------------------------
transcript
Score: 0.6141
scourse? 
 10:49: Right? 
 10:50: If your algorithm maximizes engagement by making people angry, it is not trustworthy. 
 10:54: Finally, requirement 7, accountability. 
 10:58: This brings us to audi
--------------------------------------------------
transcript
Score: 0.5789
5: It's the code plus the humans plus the laws plus the environment. 
 14:49: But there's a philosophical nugget at the end of the document that really stuck with me. 
 14:54: It says, trust is someth
--------------------------------------------------


In [18]:
baseline_results = retrieve(
    "What are the requirements for trustworthy AI?",
    top_k=5
)

for i, result in enumerate(baseline_results, start=1):
    print(f"Rank {i}")
    print(f"Source: {result['source']}")
    print(f"Score: {result['score']:.4f}")
    print(result['text'][:200])
    print("-" * 50)

Rank 1
Source: pdf
Score: 0.7614
 of Trustworthy  AI, via a list of seven 
requirements that should be met, building on the principles outlined in Chapter I. In addition, available  technical 
and non -technical methods are introduce
--------------------------------------------------
Rank 2
Source: pdf
Score: 0.7125
the trustworthiness of the AI system itself, but requires a holistic and systemic approach , encompassing  the 
trustworthiness of all actors and processes that are part of the system’s socio -technic
--------------------------------------------------
Rank 3
Source: pdf
Score: 0.7057
ith an AI system.  
 Facilitate the traceability and auditability of AI systems, particularly in critical contexts or situations.  
 Involve  stakeholders throughout the AI system ’s life cycle . Fo
--------------------------------------------------
Rank 4
Source: pdf
Score: 0.7045
ent principles and requirements. 
Continuously identify, evaluate, document and communicate these trade -offs an

In [19]:
def relevance_score(query, chunk_text):
    prompt = f"""
Rate how relevant the following text is to the query.

Query:
{query}

Text:
{chunk_text}

Respond with only a number between 1 and 10.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return float(response.choices[0].message.content.strip())

In [20]:
score = relevance_score(
    "What are the requirements for trustworthy AI?",
    baseline_results[0]["text"]
)

print(score)

9.0


In [21]:
for result in baseline_results:
    result["relevance_score"] = relevance_score(
        "What are the requirements for trustworthy AI?",
        result["text"]
    )

for result in baseline_results:
    print(
        f"Similarity: {result['score']:.4f} | "
        f"Relevance: {result['relevance_score']}"
    )

Similarity: 0.7614 | Relevance: 10.0
Similarity: 0.7125 | Relevance: 10.0
Similarity: 0.7057 | Relevance: 8.0
Similarity: 0.7045 | Relevance: 8.0
Similarity: 0.7036 | Relevance: 10.0


In [22]:
reranked_results = sorted(
    baseline_results,
    key=lambda x: x["relevance_score"],
    reverse=True
)

for i, result in enumerate(reranked_results, start=1):
    print(
        f"Rank {i} | "
        f"Similarity: {result['score']:.4f} | "
        f"Relevance: {result['relevance_score']}"
    )

Rank 1 | Similarity: 0.7614 | Relevance: 10.0
Rank 2 | Similarity: 0.7125 | Relevance: 10.0
Rank 3 | Similarity: 0.7036 | Relevance: 10.0
Rank 4 | Similarity: 0.7057 | Relevance: 8.0
Rank 5 | Similarity: 0.7045 | Relevance: 8.0
